In [1]:
import numpy as np
import pandas as pd
 
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
 
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = 'notebook_connected'
 
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
 
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
import xgboost as xgb

C:\Users\Lehma\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
 
NETFLIX_COLOR = '#E50914'
DISNEY_COLOR  = '#113CCF'
PALETTE = {'Netflix': NETFLIX_COLOR, 'Disney': DISNEY_COLOR}
 
RANDOM_STATE = 42
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

In [4]:
#Load and Inspect the Data
df = pd.read_csv('tv_movie_shows.csv')

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
seasons_per_year_since_release,6528,69.91
num_seasons,6528,69.91
maturity_x_duration,2885,30.90
director,2855,30.57
duration_minutes,2810,30.09
country,962,10.30
cast,938,10.04
freshness_x_maturity,175,1.87
year_added,93,1.00
day_of_week_added,93,1.00


In [5]:
#Clean the Data

#Parse the time data to real date-time
df['date_added_parsed'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')

# Ordinal encoding for maturity rating (informative order, not just one-hot)
rating_order = ['TV-Y', 'TV-Y7', 'TV-Y7-FV', 'G', 'TV-G', 'PG', 'TV-PG',
                'PG-13', 'TV-14', 'R', 'NC-17', 'TV-MA', 'NR', 'UR']
df['rating'] = df['rating'].astype('category')

# Clean platform / type as categories for memory + consistent ordering
df['platform'] = pd.Categorical(df['platform'], categories=['Netflix', 'Disney'])
df['type'] = pd.Categorical(df['type'], categories=['Movie', 'TV Show'])


print("Date range covered:", df['date_added_parsed'].min().date(), "to", df['date_added_parsed'].max().date())
print("\\nPlatform counts:")
print(df['platform'].value_counts())
print("\\nType counts:")
print(df['type'].value_counts())

Date range covered: 2008-01-01 to 2021-11-26
\nPlatform counts:
platform
Netflix    7926
Disney     1412
Name: count, dtype: int64
\nType counts:
type
Movie      6528
TV Show    2810
Name: count, dtype: int64
